## Navajo Code Talker

This notebook demonstrates how to encode/decode English (or other data) in Navajo with perfect round trip accuracy.

The idea comes from this [comic strip from XKCD](https://xkcd.com/257/).

In [16]:
from dataclasses import dataclass
import re

In [17]:
@dataclass
class NavajoCodeTalker:
    one_word: str = "a'la'ih"
    zero_word: str = "do'neh'lini"
    bit_separator: str = " "
    byte_separator: str = ", "
    ending: str = "."
    capitalize_first: bool = True

    def talk(self, string: str):
        utf8_encoded_bytes = string.encode('utf-8')

        def encode_bit(bit):
            return self.one_word if bit == '1' else self.zero_word

        # encode each byte in Navajo
        bytes = []
        for byte in utf8_encoded_bytes:
            # a string of length 8 containing characters '1' and '0'
            binary_string = bin(byte)[2:].zfill(8)

            byte = self.bit_separator.join(encode_bit(bit) for bit in binary_string)
            bytes.append(byte)
            
        
        # Join and format the string
        encoded = (self.byte_separator.join(bytes) + self.ending)
        if self.capitalize_first:
            encoded = encoded.capitalize()

        return encoded
    
    def listen(self, string: str):
        # Create a regex pattern that is case-insensitive and matches the specified one_word and zero_word
        one_word = re.escape(self.one_word)
        zero_word = re.escape(self.zero_word)
        pattern = re.compile(rf"({one_word}|{zero_word})", re.IGNORECASE)
        
        binary_strings = []
        
        # Find all matches of the one_word and zero_word
        for match in pattern.findall(string):
            # Convert the matched words to '1' or '0'
            bit = '1' if match.lower() == self.one_word.lower() else '0'
            binary_strings.append(bit)
        
        # Group binary strings into bytes
        bytes_list = [''.join(binary_strings[i:i+8]) for i in range(0, len(binary_strings), 8)]
        
        # Convert binary strings to bytes and decode from UTF-8
        decoded_string = ''.join([chr(int(byte, 2)) for byte in bytes_list if len(byte) == 8])
        
        return decoded_string

In [18]:
# Navajo code talker example:
code_talker = NavajoCodeTalker(byte_separator=",\n")
print(code_talker.talk("Hello!"))

Do'neh'lini a'la'ih do'neh'lini do'neh'lini a'la'ih do'neh'lini do'neh'lini do'neh'lini,
do'neh'lini a'la'ih a'la'ih do'neh'lini do'neh'lini a'la'ih do'neh'lini a'la'ih,
do'neh'lini a'la'ih a'la'ih do'neh'lini a'la'ih a'la'ih do'neh'lini do'neh'lini,
do'neh'lini a'la'ih a'la'ih do'neh'lini a'la'ih a'la'ih do'neh'lini do'neh'lini,
do'neh'lini a'la'ih a'la'ih do'neh'lini a'la'ih a'la'ih a'la'ih a'la'ih,
do'neh'lini do'neh'lini a'la'ih do'neh'lini do'neh'lini do'neh'lini do'neh'lini a'la'ih.


In [10]:
# Listening example
code_listener = NavajoCodeTalker()
print(code_listener.listen("""
Do'neh'lini a'la'ih do'neh'lini do'neh'lini a'la'ih do'neh'lini do'neh'lini a'la'ih,
do'neh'lini a'la'ih a'la'ih a'la'ih do'neh'lini a'la'ih do'neh'lini do'neh'lini,
do'neh'lini do'neh'lini a'la'ih do'neh'lini do'neh'lini do'neh'lini do'neh'lini do'neh'lini,
do'neh'lini a'la'ih a'la'ih a'la'ih do'neh'lini a'la'ih a'la'ih a'la'ih,
do'neh'lini a'la'ih a'la'ih do'neh'lini a'la'ih a'la'ih a'la'ih a'la'ih,
do'neh'lini a'la'ih a'la'ih a'la'ih do'neh'lini do'neh'lini a'la'ih do'neh'lini,
do'neh'lini a'la'ih a'la'ih do'neh'lini a'la'ih do'neh'lini a'la'ih a'la'ih,
do'neh'lini a'la'ih a'la'ih do'neh'lini do'neh'lini a'la'ih do'neh'lini a'la'ih,
do'neh'lini a'la'ih a'la'ih do'neh'lini do'neh'lini a'la'ih do'neh'lini do'neh'lini,
do'neh'lini do'neh'lini a'la'ih do'neh'lini do'neh'lini do'neh'lini do'neh'lini a'la'ih.
"""))


It worked!


In [19]:
message = "This sentence is in English."
encoded = code_talker.talk(message)
decoded = code_talker.listen(encoded)

assert message == decoded

print(decoded)


This sentence is in English.
